
# Chapter 12 : Context Window Analyzer


Same question
+
Increasing amounts of context

What is a Token?
A token is a small piece of text that an LLM processes as input or generates as output.
A token is not necessarily a word.

| Token Type | What it means | Example / Usage |
|---|---|---|
| **Input Tokens** | Tokens sent **to the LLM** as the prompt/context | Your question, system prompt, conversation history |
| **Output Tokens** | Tokens generated **by the LLM** in its response | The answer generated by the model |
| **Reasoning Tokens** | Tokens attributed to the model's internal reasoning, when the provider reports them | Your metadata showed **182 reasoning tokens** |
| **Total Tokens** | Total tokens consumed for the request | `Input + Completion` |
| **Completion Tokens** | Total tokens generated during completion; may include reasoning tokens depending on the provider's accounting | Your metadata showed **605** |
| **Word/Subword Tokens** | Pieces representing whole words or parts of words | `un` + `believ` + `able` |
| **Punctuation Tokens** | Tokens representing punctuation or punctuation-related text | `.`, `,`, `?`, `!` |
| **Whitespace-related Tokens** | Tokens that can include spaces together with text | `" hello"` |
| **Special Tokens** | Structural tokens used by the tokenizer/model | Beginning/end markers, message/role markers, etc. |

                 REQUEST
                     │
                     ▼
             generate_content()
                     │
                     ▼
              Open AI Model
                     │
                     ▼
                  RESPONSE
                     │
        ┌────────────┼─────────────┐
        ▼            ▼             ▼
      Text       Candidates    Token Usage
        │            │             │
        ▼            ▼             ▼
 response.text   candidate[0]   usage_metadata

| Method | Async? | Requests | Response type | Use case |
|---|---|---|---|---|
| `invoke()` | ❌ No | One | Complete response | One LLM request |
| `ainvoke()` | ✅ Yes | One | Complete response | One async LLM request |
| `batch()` | ❌ No | Many | Complete responses | Multiple independent requests |
| `abatch()` | ✅ Yes | Many | Complete responses | Multiple independent async requests |
| `stream()` | ❌ No | One | Response chunks | Show response progressively |
| `astream()` | ✅ Yes | One | Response chunks | Async progressive response |

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()


os.environ["GROQ_API_KEY"] = "YOUR API"


In [2]:
# Setting up environment variable

def setup_environment():
  """ Loads environment variables and check for the required API key"""

  load_dotenv()
  if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is not set")


In [3]:
%pip install -U langchain langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.3
    Uninstalling langchain-core-1.6.3:
      Successfully uninstalled langchain-core-1.6.3
  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.0
    Uninstalling langchain-1.4.0:
      Successfully uninstalled langchain-1.4.0


In [4]:
!pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 7.8 MB/s eta 0:00:00


In [5]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],

)

In [6]:
model = "openai/gpt-oss-20b"

In [7]:
# STEP 1: Create some base text

In [10]:
base_text = """
Retrieval Augmented Generation (RAG) is a technique used to improve
LLM responses by retrieving relevant information from external data.

A typical RAG system contains several steps. Documents are first loaded
and divided into smaller chunks. These chunks are converted into
embeddings and stored in a vector database.

When a user asks a question, the question is converted into an embedding.
The system then searches the vector database for relevant chunks.

The retrieved information is then provided to the language model along
with the user's question. The model uses this information to generate
a grounded response.
"""

In [11]:
# Step 2 — Create progressively larger contexts
# We'll repeat the text to simulate increasing context.

In [12]:
contexts = {
    "Small": base_text,
    "Medium": base_text * 5,
    "Large": base_text * 20,
    "Very Large": base_text * 50
}

In [14]:
question = "What is RAG and how does it work?"


In [23]:
for i, context in contexts.items():
  prompt = f""" Context : {Context}
  Question :
  {question}
  Answer using the context above.
  """
  response = llm.invoke(prompt)
  print("CONTEXT:", name)

  usage = response.usage_metadata
  print(usage)

CONTEXT: Very Large
{'input_tokens': 102, 'output_tokens': 1308, 'total_tokens': 1410, 'input_token_details': {}, 'output_token_details': {'reasoning': 836}}
CONTEXT: Very Large
{'input_tokens': 102, 'output_tokens': 803, 'total_tokens': 905, 'input_token_details': {}, 'output_token_details': {'reasoning': 384}}
CONTEXT: Very Large
{'input_tokens': 102, 'output_tokens': 1427, 'total_tokens': 1529, 'input_token_details': {}, 'output_token_details': {'reasoning': 669}}
CONTEXT: Very Large
{'input_tokens': 102, 'output_tokens': 921, 'total_tokens': 1023, 'input_token_details': {}, 'output_token_details': {'reasoning': 520}}


In [21]:
results = []

for name, context in contexts.items():

    prompt = f"""
Context:

{context}

Question:
{question}

Answer using the context above.
"""

    response = llm.invoke(prompt)

    usage = response.usage_metadata

    results.append({
        "Context": name,
        "Input Tokens": usage["input_tokens"],
        "Output Tokens": usage["output_tokens"],
        "Total Tokens": usage["total_tokens"]
    })

In [22]:
import pandas as pd

df = pd.DataFrame(results)

df

,Context,Input Tokens,Output Tokens,Total Tokens
0,Small,209,399,608
1,Medium,677,337,1014
2,Large,2432,382,2814
3,Very Large,5942,334,6276


In [26]:
for name, context in contexts.items():

    prompt = f"""
Context:

{context}

Question:
{question}

Answer using the context above.
"""

    response = llm.invoke(prompt)

    usage = response.usage_metadata

    print("\n" + "=" * 70)
    print("CONTEXT:", name)
    print("=" * 70)

    print("Input tokens :", usage["input_tokens"])
    print("Output tokens:", usage["output_tokens"])
    print("Total tokens :", usage["total_tokens"])

    print("\nAnswer:")
    print(usage)


CONTEXT: Small
Input tokens : 209
Output tokens: 314
Total tokens : 523

Answer:
{'input_tokens': 209, 'output_tokens': 314, 'total_tokens': 523, 'input_token_details': {}, 'output_token_details': {'reasoning': 103}}

CONTEXT: Medium
Input tokens : 677
Output tokens: 294
Total tokens : 971

Answer:
{'input_tokens': 677, 'output_tokens': 294, 'total_tokens': 971, 'input_token_details': {}, 'output_token_details': {'reasoning': 31}}

CONTEXT: Large
Input tokens : 2432
Output tokens: 287
Total tokens : 2719

Answer:
{'input_tokens': 2432, 'output_tokens': 287, 'total_tokens': 2719, 'input_token_details': {}, 'output_token_details': {'reasoning': 105}}

CONTEXT: Very Large
Input tokens : 5942
Output tokens: 393
Total tokens : 6335

Answer:
{'input_tokens': 5942, 'output_tokens': 393, 'total_tokens': 6335, 'input_token_details': {}, 'output_token_details': {'reasoning': 157}}


# Measuring Latency

In [27]:
results = []

for name, context in contexts.items():

    prompt = f"""
Context:

{context}

Question:
{question}

Answer using the context above.
"""

    response = llm.invoke(prompt)

    usage = response.usage_metadata
    metadata = response.response_metadata

    token_usage = metadata.get("token_usage", {})

    results.append({
        "Context": name,
        "Input Tokens": usage.get("input_tokens"),
        "Output Tokens": usage.get("output_tokens"),
        "Total Tokens": usage.get("total_tokens"),
        "Total Time": token_usage.get("total_time")
    })

In [29]:
# As context size increases total time also increases

In [28]:
df = pd.DataFrame(results)

df

,Context,Input Tokens,Output Tokens,Total Tokens,Total Time
0,Small,209,238,447,0.260966
1,Medium,677,245,922,0.305423
2,Large,2432,279,2711,0.469088
3,Very Large,5942,310,6252,0.638256
